In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import joblib
import ccxt
from sklearn.feature_selection import mutual_info_regression
import gc
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [2]:
data = pd.read_parquet(r"C:\Quant\Merged_kimp.parquet")


In [3]:
data.head()

,market,timeframe,timestamp_utc,timestamp_kst,open_u,high_u,low_u,close_u,volume_u,value,...,open_time,open_b,high_b,low_b,close_b,volume_b,taker_buy_base_volume,open_time_kst,market_fx,kimp_real
0,KRW-BTC,minutes/1,2026-02-26 05:13:00,2026-02-26 14:13:00,98333000.0,98410000.0,98318000.0,98410000.0,2.894298,2.846193e+08,...,2026-02-26 05:13:00+00:00,68297.242188,68309.617188,68297.226562,68309.601562,2.58417,1.213030,2026-02-26 14:13:00,1430.122816,0.735871
1,KRW-BTC,minutes/1,2026-02-26 05:14:00,2026-02-26 14:14:00,98410000.0,98447000.0,98400000.0,98411000.0,0.781258,7.688261e+07,...,2026-02-26 05:14:00+00:00,68309.601562,68322.656250,68301.570312,68301.570312,4.54242,0.742770,2026-02-26 14:14:00,1430.126559,0.748476
2,KRW-BTC,minutes/1,2026-02-26 05:15:00,2026-02-26 14:15:00,98446000.0,98453000.0,98359000.0,98360000.0,1.356775,1.335355e+08,...,2026-02-26 05:15:00+00:00,68301.562500,68326.289062,68229.062500,68229.070312,14.27996,6.972280,2026-02-26 14:15:00,1430.130302,0.803000
3,KRW-BTC,minutes/1,2026-02-26 05:16:00,2026-02-26 14:16:00,98360000.0,98373000.0,98229000.0,98229000.0,3.428919,3.371728e+08,...,2026-02-26 05:16:00+00:00,68229.070312,68230.000000,68149.250000,68164.140625,14.27097,3.426340,2026-02-26 14:16:00,1430.134045,0.764375
4,KRW-BTC,minutes/1,2026-02-26 05:17:00,2026-02-26 14:17:00,98230000.0,98230000.0,98139000.0,98200000.0,2.266259,2.224902e+08,...,2026-02-26 05:17:00+00:00,68164.492188,68191.429688,68127.578125,68191.421875,36.75370,17.216499,2026-02-26 14:17:00,1430.137788,0.694062


In [4]:
data.columns

Index(['market', 'timeframe', 'timestamp_utc', 'timestamp_kst', 'open_u',
       'high_u', 'low_u', 'close_u', 'volume_u', 'value', 'symbol',
       'open_time', 'open_b', 'high_b', 'low_b', 'close_b', 'volume_b',
       'taker_buy_base_volume', 'open_time_kst', 'market_fx', 'kimp_real'],
      dtype='object')

In [ ]:
# 1. 환경 및 분석 대상 설정
exchange = ccxt.binance()
base_market = 'BTC/USDT'
target_markets = ['ETH/USDT', 'SOL/USDT', 'XRP/USDT', 'DOGE/USDT']
all_markets = [base_market] + target_markets

# 2. 가격 데이터 수집 함수 (최근 1000개의 1시간봉 기준)
def fetch_close_prices(symbols, timeframe='1h', limit=1000):
    data = {}
    for sym in symbols:
        print(f"[{sym}] 데이터 수집 중")
        ohlcv = exchange.fetch_ohlcv(sym, timeframe, limit=limit)
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        data[sym] = df['close'] # 종가(Close)만 추출
    return pd.DataFrame(data).dropna()

# 3. 가격 데이터 로드 및 수익률(returns_data) 생성
price_data = fetch_close_prices(all_markets)
returns_data = price_data.pct_change().dropna()

# 4. 베타(Beta) 지수 계산 로직 적용
print("베타 지수 계산 중")

# 비트코인(시장 기준) 수익률의 분산(Variance) 계산
btc_variance = returns_data[base_market].var()

# 전체 자산의 공분산 행렬에서 비트코인과의 공분산(Covariance) 추출
covariances = returns_data.cov()[base_market][target_markets]

# 베타(Beta) = 공분산 / 비트코인 분산
beta_values = covariances / btc_variance

print(f"\n=== {base_market} 대비 알트코인 베타(Beta) 지수 =")
print(beta_values)


[BTC/USDT] 데이터 수집 중
[ETH/USDT] 데이터 수집 중
[SOL/USDT] 데이터 수집 중
[XRP/USDT] 데이터 수집 중
[DOGE/USDT] 데이터 수집 중
베타 지수 계산 중

=== BTC/USDT 대비 알트코인 베타(Beta) 지수 =
ETH/USDT     1.160438
SOL/USDT     1.193109
XRP/USDT     0.976556
DOGE/USDT    1.048027
Name: BTC/USDT, dtype: float64


## 베타(Beta) 지수 해석 가이드
$\beta = 1$: 비트코인과 완전히 동일한 변동성을 가집니다. (비트코인이 1% 오르면 알트코인도 1% 오름)\
$\beta > 1$ (고위험/고수익): 비트코인보다 더 민감하게 움직입니다. 예를 들어 베타가 1.5라면, 비트코인이 1% 오를 때 알트코인은 1.5% 오르고, 반대로 1% 떨어질 때는 1.5% 떨어집니다. 보통 시가총액이 낮은 알트코인이나 밈코인들이 1 이상의 높은 베타를 가짐.\
$0 < \beta < 1$ (저위험): 비트코인과 같은 방향으로 움직이지만, 변동폭은 더 작음
$\beta \approx 0$: 비트코인의 움직임과 무관하게 독자적인 가격 흐름을 가짐. (앞서 본 ENSO 같은 코인이 여기에 해당할 확률이 높음)\
$\beta < 0$ (역방향): 비트코인과 반대로 움직이는 자산

In [ ]:
# 1. 베타 지수 정의
beta_values = {
    'KRW-BTC': 1.0, 'KRW-ETH': 1.161173, 'KRW-SOL': 1.193111, 
    'KRW-XRP': 0.979747, 'KRW-DOGE': 1.047770
}

# 2. 피처 생성 함수 (수치 안정성 강화)
def generate_missing_features(df):
    print("피처 연산을 시작합니다... (3,300만 행 대응)")
    df = df.sort_values(['market', 'timestamp_kst'])
    grouped = df.groupby('market')

    # 기초 지표
    df['kimp_velocity'] = grouped['kimp_real'].diff()
    df['ret_lag_1'] = grouped['close_u'].pct_change(1)
    df['ret_lag_5'] = grouped['close_u'].pct_change(5)
    
    # 변동성 및 RSI
    df['volatility_30m'] = grouped['close_u'].transform(lambda x: x.pct_change().rolling(window=30).std())
    
    def calc_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        return 100 - (100 / (1 + (gain / (loss + 1e-9)) + 1e-9))
    df['rsi_14'] = grouped['close_u'].transform(calc_rsi)

    # 이격도 및 거래량 서지
    df['sma_diff'] = grouped['close_u'].transform(lambda x: (x / x.rolling(window=20).mean()) - 1)
    df['vol_surge'] = df['volume_u'] / (grouped['volume_u'].transform(lambda x: x.rolling(window=30).mean()) + 1e-9)
    df['kimp_gap'] = df['kimp_real'] - grouped['kimp_real'].transform(lambda x: x.rolling(window=30).mean())

    # BTC 도미넌스
    btc_series = df[df['market'] == 'KRW-BTC'].groupby('timestamp_kst')['value'].mean()
    total_val_series = df.groupby('timestamp_kst')['value'].transform('sum')
    df['btc_vol_dom'] = df['timestamp_kst'].map(btc_series) / (total_val_series + 1e-9)

    # 모든 결측치를 0으로 우선 채움 (중간 단계)
    df = df.fillna(0)
    # 무한대(inf) 값이 생겼을 경우를 대비해 처리
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 3. 상관관계 분석 함수 (NaN 제거 강화)
def analyze_advanced_features(df, features, target='target_return_30m', n_sample=1000000):
    print(f"상관관계 분석 준비 중...")
    
    # 타겟 데이터 생성 (30분 뒤 수익률)
    df['target_return_30m'] = df.groupby('market')['close_u'].shift(-30).pct_change(30) * 100
    
    # [핵심] 분석 직전에 NaN이 하나라도 있는 행은 완전히 제거
    clean_df = df.dropna(subset=features + [target])
    
    # 무한대 값 제거
    clean_df = clean_df.replace([np.inf, -np.inf], np.nan).dropna(subset=features + [target])
    
    print(f"상관관계 분석 시작 (정제 후 샘플링: {n_sample}행)")
    sample = clean_df.sample(n=min(n_sample, len(clean_df)), random_state=42)
    
    X = sample[features].astype('float32')
    y = sample[target].astype('float32')
    
    # 선형 점수
    linear_score = X.corrwith(y).abs()
    
    # 비선형 점수 (이제 NaN이 없으므로 에러 안 남)
    mi_scores = mutual_info_regression(X, y, discrete_features=False, random_state=42)
    non_linear_score = pd.Series(mi_scores, index=features)
    
    report = pd.DataFrame({
        'Linear_Power': linear_score / (linear_score.max() + 1e-9),
        'NonLinear_Power': non_linear_score / (non_linear_score.max() + 1e-9)
    })
    report['Total_Score'] = report['Linear_Power'] + report['NonLinear_Power']
    return report.sort_values(by='Total_Score', ascending=False)

# --- 메인 실행 ---
# 1. 베타 지수 추가
data['beta_index'] = data['market'].map(beta_values).fillna(1.0)

# 2. 피처 생성
data = generate_missing_features(data)

# 3. 분석 보고서 출력
candidate_features = [
    'beta_index', 'close_b', 'kimp_real', 'kimp_velocity', 'market_fx', 
    'volatility_30m', 'taker_buy_base_volume', 'btc_vol_dom',
    'rsi_14', 'sma_diff', 'vol_surge', 'kimp_gap', 'ret_lag_1', 'ret_lag_5'
]

feature_report = analyze_advanced_features(data, candidate_features)

print("\n[최종 피처 영향력 분석 보고서]")
print("-" * 65)
print(feature_report)
print("-" * 65)

gc.collect()

피처 연산을 시작합니다... (3,300만 행 대응)
상관관계 분석 준비 중...


C:\Users\user\AppData\Local\Temp\ipykernel_19092\1231190132.py:55: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['target_return_30m'] = df.groupby('market')['close_u'].shift(-30).pct_change(30) * 100


상관관계 분석 시작 (정제 후 샘플링: 1000000행)

[최종 피처 영향력 분석 보고서]
-----------------------------------------------------------------
                       Linear_Power  NonLinear_Power  Total_Score
sma_diff                   1.000000         0.550981     1.550981
ret_lag_1                  0.580897         0.891207     1.472104
ret_lag_5                  0.132699         1.000000     1.132699
close_b                    0.038548         0.987997     1.026545
kimp_gap                   0.851499         0.074536     0.926034
beta_index                 0.096892         0.408385     0.505277
volatility_30m             0.047107         0.346178     0.393285
taker_buy_base_volume      0.014670         0.242940     0.257610
kimp_velocity              0.086811         0.157694     0.244505
market_fx                  0.087560         0.146479     0.234039
rsi_14                     0.040140         0.169862     0.210003
kimp_real                  0.044177         0.024695     0.068872
vol_surge               

345

In [ ]:
# ---------------------------------------------------------------------------
# [1] 실전형 손익비 백테스트 함수 정의 (NaN 및 타입 오류 방지)
# ---------------------------------------------------------------------------
def run_rr_backtest(probs, rets, fee=0.05, tp=1.0, sl=-0.5):
    # 상위 0.5% 타점 추출
    th = np.percentile(probs, 99.5)
    indices = np.where(probs >= th)[0]
    
    # rets 데이터가 Series일 경우 values로 처리
    rets_values = rets.values if hasattr(rets, 'values') else rets
    trades = rets_values[indices]
    
    final_rets = []
    for r in trades:
        if np.isnan(r): 
            continue
        if r >= tp: 
            final_rets.append(tp - fee)   # 익절 성공
        elif r <= sl: 
            final_rets.append(sl - fee)   # 손절 발생
        else: 
            final_rets.append(r - fee)    # 30분 타임컷 종료
            
    return np.array(final_rets)

# ---------------------------------------------------------------------------
# [2] 데이터 정제 및 타겟 생성
# ---------------------------------------------------------------------------
print("데이터 정제 및 학습 준비 중...")

# 타겟(target_rr) 생성: 30분 뒤 수익률이 1.0% 이상인 지점
data['target_rr'] = (data['target_return_30m'] >= 1.0).astype(np.int8)

# 분석 결과 기반 상위 10개 피처 선정
final_selected_features = [
    'sma_diff', 'ret_lag_1', 'ret_lag_5', 'close_b', 'kimp_gap',
    'beta_index', 'volatility_30m', 'taker_buy_base_volume', 'kimp_velocity', 'market_fx'
]

# 학습에 방해되는 결측치 제거
clean_data = data.dropna(subset=final_selected_features + ['target_rr', 'target_return_30m'])

# 학습/테스트 데이터셋 분할 (2025-01-01 기준)
train_idx = clean_data['timestamp_kst'] < '2025-01-01'
X_train = clean_data.loc[train_idx, final_selected_features].astype('float32')
y_train = clean_data.loc[train_idx, 'target_rr'].astype('int8')
X_test = clean_data.loc[~train_idx, final_selected_features].astype('float32')
y_test = clean_data.loc[~train_idx, 'target_rr'].astype('int8')
actual_rets_test = clean_data.loc[~train_idx, 'target_return_30m']

# 클래스 불균형 가중치 계산
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"학습 준비 완료. 총 피처: {len(final_selected_features)}개")

# ---------------------------------------------------------------------------
# [3] 3대 알고리즘 GPU 학습 (최신 API 규격 적용)
# ---------------------------------------------------------------------------
print("알고리즘별 GPU 학습을 시작합니다...")

# LightGBM (GPU)
lgbm = lgb.LGBMClassifier(
    device='gpu', n_estimators=1000, scale_pos_weight=pos_weight, 
    learning_rate=0.01, verbose=-1
)
lgbm.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(100)])

# XGBoost (GPU / 최신 규격)
xgboost = xgb.XGBClassifier(
    tree_method='hist', device='cuda', n_estimators=1000, 
    scale_pos_weight=pos_weight, learning_rate=0.01, early_stopping_rounds=50
)
xgboost.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# CatBoost (GPU)
catboost = CatBoostClassifier(
    task_type='GPU', iterations=1000, scale_pos_weight=pos_weight, 
    learning_rate=0.01, verbose=100, early_stopping_rounds=50
)
catboost.fit(X_train, y_train, eval_set=(X_test, y_test))

# ---------------------------------------------------------------------------
# [4] 성과 보고서 출력 함수 정의 및 실행
# ---------------------------------------------------------------------------
def get_final_performance(models, X_test_data, actual_returns):
    perf = {}
    for name, model in models.items():
        # 확률 예측
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_test_data)[:, 1]
        else:
            probs = model.predict(X_test_data)
        
        # 정의된 백테스트 함수 호출
        res = run_rr_backtest(probs, actual_returns) 
        
        if len(res) > 0:
            perf[name] = {
                '거래 횟수': len(res),
                '승률': f"{(res > 0).mean()*100:.2f}%",
                '누적 수익률': f"{res.sum():.2f}%",
                '평균 수익률': f"{res.mean():.4f}%"
            }
    return pd.DataFrame(perf).T

print("\n" + "="*80)
print("5종 코인 통합 모델 최종 실전 성과 비교")
print("="*80)
models_dict = {'LGBM': lgbm, 'XGBoost': xgboost, 'CatBoost': catboost}
final_report = get_final_performance(models_dict, X_test, actual_rets_test)
print(final_report)
print("="*80)

gc.collect()

데이터 정제 및 학습 준비 중...
학습 준비 완료. 총 피처: 10개
알고리즘별 GPU 학습을 시작합니다...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[5]	valid_0's binary_logloss: 0.105913
0:	learn: 0.6879850	test: 0.6878762	best: 0.6878762 (0)	total: 198ms	remaining: 3m 17s
100:	learn: 0.4832886	test: 0.4791798	best: 0.4791798 (100)	total: 13.6s	remaining: 2m 1s
200:	learn: 0.4524219	test: 0.4483143	best: 0.4483143 (200)	total: 26.8s	remaining: 1m 46s
300:	learn: 0.4433403	test: 0.4399727	best: 0.4399727 (300)	total: 39.9s	remaining: 1m 32s
400:	learn: 0.4388396	test: 0.4369537	best: 0.4369537 (400)	total: 53.1s	remaining: 1m 19s
500:	learn: 0.4359386	test: 0.4355485	best: 0.4355462 (493)	total: 1m 6s	remaining: 1m 6s
600:	learn: 0.4337735	test: 0.4340882	best: 0.4340882 (600)	total: 1m 19s	remaining: 52.9s
700:	learn: 0.4319771	test: 0.4333043	best: 0.4333043 (700)	total: 1m 33s	remaining: 39.7s
800:	learn: 0.4305930	test: 0.4327301	best: 0.4327301 (800)	total: 1m 46s	remai

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\core.py:774: UserWarning: [23:42:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


          거래 횟수      승률     누적 수익률   평균 수익률
LGBM      33218  74.06%  15742.70%  0.4739%
XGBoost   29816  81.72%  18134.37%  0.6082%
CatBoost  29816  78.12%  16417.83%  0.5506%


831

In [9]:
import joblib

# 최강 모델 XGBoost 저장
model_path = "C:/Quant/best_xgb_integrated_model.json"
xgboost.save_model(model_path)
print(f"최종 모델이 {model_path}에 저장되었습니다.")

최종 모델이 C:/Quant/best_xgb_integrated_model.json에 저장되었습니다.
